In [0]:
%run ../00-common/config

In [0]:
from pyspark.sql import functions as F

customers = spark.table(f"{catalog_name}.{silver_schema}.customers")

# dim_customer: customer_id eshte PK, i njejti person merr id te re sahere qe porosit
# customer_unique_id identifikon personin real (pra i njejti qe mund te kete porositur disa here)
# dim_customer = customers.select(
#     "customer_unique_id",
#     "customer_zip_code_prefix",
#     "customer_city",
#     "customer_state"
# )

# PK: customer_unique_id
dim_customer = (
    customers
    .filter(F.col("customer_unique_id").isNotNull())
    .select("customer_unique_id")
    .dropDuplicates(["customer_unique_id"])
)

(dim_customer.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{gold_schema}.dim_customer"))
print(f"Wrote {dim_customer.count():,} customers")